In [1]:
import pandas as pd

In [2]:
raw_df = pd.read_csv("drx dataset.csv")

# Keep only records with the fields used in the modelling workflow.
# The source CSV also contains Atmosphere-only metadata rows, which are excluded here.
required_columns = ["Composition", "DRX label", "Method", "t (h)", "T (°C)"]
initial_row_count = len(raw_df)
drx_df = raw_df.dropna(subset=required_columns).loc[:, required_columns].copy()
print(f"Loaded {initial_row_count} source rows; removed {initial_row_count - len(drx_df)} incomplete rows; {len(drx_df)} modelling records remain.")
drx_df

Loaded 182 source rows; removed 9 incomplete rows; 173 modelling records remain.


,Composition,DRX label,Method,t (h),T (°C)
0,Li1.05Mn0.85Ti0.1O2,unsuccessful,Solid-state,0.33,1100.0
1,Li1.1Mn0.7Ti0.2O2,unsuccessful,Solid-state,0.33,1100.0
2,Li1.15Mn0.55Ti0.3O2,unsuccessful,Solid-state,0.33,1100.0
3,Li1.2Mn0.4Ti0.4O2,successful,Solid-state,10.00,1000.0
4,Li1.2Cr0.1Mn0.3Ti0.4O2,successful,Solid-state,10.00,1000.0
...,...,...,...,...,...
168,LiFeO2,unsuccessful,Hydrothermal,8.00,220.0
169,LiCo0.5Ti0.5O2,successful,Sol-gel,10.00,550.0
170,LiCo0.5Ti0.5O2,successful,Sol-gel,10.00,600.0
171,LiCo0.5Ti0.5O2,successful,Sol-gel,10.00,650.0


In [3]:
print(drx_df.dtypes)

Composition     object
DRX label       object
Method          object
t (h)          float64
T (°C)         float64
dtype: object


In [4]:
drx_df["DRX label"] = drx_df["DRX label"].str.lower()

In [5]:
drx_df["DRX label"]

0      unsuccessful
1      unsuccessful
2      unsuccessful
3        successful
4        successful
           ...     
168    unsuccessful
169      successful
170      successful
171      successful
172      successful
Name: DRX label, Length: 173, dtype: object

In [6]:
drx_df["DRX label"] = drx_df["DRX label"].map({"successful": 1, "unsuccessful": 0})
# map(...) → 按照括号里的对照表替换值

In [7]:
drx_df["DRX label"]

0      0
1      0
2      0
3      1
4      1
      ..
168    0
169    1
170    1
171    1
172    1
Name: DRX label, Length: 173, dtype: int64

In [8]:
print(drx_df["Method"].value_counts())
print(drx_df["t (h)"].value_counts())

Method
Solid-state        105
Sol-gel             29
Mechanochemical     22
Molten-salt         12
Hydrothermal         5
Name: count, dtype: int64
t (h)
12.000    38
10.000    33
2.000     16
20.000    15
16.000     8
24.000     7
1.000      7
36.000     7
8.000      6
0.083      4
96.000     4
0.330      3
4.000      3
0.167      3
5.000      3
6.000      3
3.000      2
0.017      2
40.000     2
0.333      2
14.000     1
7.000      1
0.500      1
0.050      1
0.003      1
Name: count, dtype: int64


In [9]:
drx_df = drx_df.rename(columns={"Method\n": "Method"})

In [10]:
drx_df["Method"] = pd.Categorical(drx_df["Method"], categories=["Hydrothermal", "Mechanochemical", "Molten-salt", "Sol-gel", "Solid-state"])
#pd.Categorical(...) pandas里的一个功能，把普通的一列数据变成category类型。括号里放两个东西：第一个是你的数据，第二个是categories=你自己指定类别。

method_categories = drx_df["Method"].cat.categories
#把这个["Hydrothermal", "Mechanochemical", "Molten-salt", "Sol-gel", "Solid-state"]列表存进一个叫method_categories的变量里

drx_df["Method"] = drx_df["Method"].cat.codes
#把类别名字按顺序换成数字

In [11]:
drx_df["Method"]

0      4
1      4
2      4
3      4
4      4
      ..
168    0
169    3
170    3
171    3
172    3
Name: Method, Length: 173, dtype: int8

In [12]:
drx_df["t (h)"] = pd.to_numeric(drx_df["t (h)"], errors="coerce")

In [13]:
print((drx_df["Method"] == -1).sum())
drx_df["t (h)"].isna().sum()
# Check category coding and missing synthesis-time values.

0


np.int64(0)

In [14]:
drx_df[drx_df["t (h)"].isna()]

,Composition,DRX label,Method,t (h),T (°C)


In [15]:
# 温度和时间的基本统计：mean, min, max, range
print("=== T (°C) ===")
print(drx_df["T (°C)"].describe())

print()
print("=== t (h) ===")
print(drx_df["t (h)"].describe())

print()
print("=== DRX label ===")
print(drx_df["DRX label"].value_counts())

=== T (°C) ===
count     173.000000
mean      788.786127
std       306.089790
min        25.000000
25%       700.000000
50%       900.000000
75%       950.000000
max      1300.000000
Name: T (°C), dtype: float64

=== t (h) ===
count    173.000000
mean      13.064023
std       15.516725
min        0.003000
25%        4.000000
50%       10.000000
75%       14.000000
max       96.000000
Name: t (h), dtype: float64

=== DRX label ===
DRX label
1    144
0     29
Name: count, dtype: int64


The final cleaned dataset contains 173 literature records and 115 unique composition formulae.
There are 144 successful records and 29 unsuccessful records (83.2% and 16.8%, respectively).
The class imbalance should be considered during model training and evaluation.

In [16]:
from collections import Counter
import re #regular expression

# 把所有Composition里的元素提取出来
all_elements = []

for composition in drx_df["Composition"]:
    # re.findall找出所有大写字母开头的元素符号
    elements = re.findall(r'[A-Z][a-z]?', str(composition))
    all_elements.extend(elements)

# 数每个元素出现几次
element_counts = Counter(all_elements)
print(element_counts)

Counter({'Li': 173, 'O': 173, 'Ti': 117, 'Mn': 76, 'Ni': 50, 'Nb': 39, 'Fe': 29, 'Mo': 14, 'V': 13, 'Co': 10, 'Zr': 10, 'Cr': 8, 'W': 4, 'Ru': 3, 'Mg': 2, 'Ta': 1})


In [17]:
# 把Composition和DRX label两列取出来，改成formula和target
cbfv_df = drx_df[["Composition", "DRX label"]].copy()
cbfv_df = cbfv_df.rename(columns={"Composition": "formula", "DRX label": "target"})
cbfv_df

,formula,target
0,Li1.05Mn0.85Ti0.1O2,0
1,Li1.1Mn0.7Ti0.2O2,0
2,Li1.15Mn0.55Ti0.3O2,0
3,Li1.2Mn0.4Ti0.4O2,1
4,Li1.2Cr0.1Mn0.3Ti0.4O2,1
...,...,...
168,LiFeO2,0
169,LiCo0.5Ti0.5O2,1
170,LiCo0.5Ti0.5O2,1
171,LiCo0.5Ti0.5O2,1


In [18]:
import sys
print(sys.executable)

c:\Users\胡扬\AppData\Local\Programs\Python\Python312\python.exe


In [19]:
import sys
!{sys.executable} -m pip install --upgrade setuptools

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.2.1-py3-none-any.whl (876 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
!{sys.executable} -m pip show setuptools

Name: setuptools
Version: 84.2.1
Summary: Most extensible Python build backend with support for C/C++ extension modules
Home-page: 
Author: 
Author-email: Python Packaging Authority <distutils-sig@python.org>
License-Expression: MIT
Location: c:\Users\胡扬\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: 
Required-by: 


In [21]:
import sys
!{sys.executable} -m pip install "setuptools<81"

  Using cached setuptools-80.10.2-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-80.10.2-py3-none-any.whl (1.1 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 84.2.1
    Uninstalling setuptools-84.2.1:
      Successfully uninstalled setuptools-84.2.1



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install pandas==2.2.3
%pip install CBFV


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd


^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  You can safely remove it manually.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.5 MB 2.8 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.5 MB 2.4 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/11.5 MB 2.5 MB/s eta 0:00:04
   ------- -------------------------------- 2.1/11.5 MB 2.7 MB/s eta 0:00:04
   --------- ------------------------------ 2.6/11.5 MB 2.7 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/11.5 MB 2.6 MB/s eta 0:00:04
   ------------- -------------------------- 3.9/11.5 MB 2.7 MB/s eta 0:00:03
   --------------- ------------------------ 4.5/11.5 MB 2.7 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.5 MB 2.6 MB/s eta 0:00:03
   ------------------ --------------------- 5.2/11.5 MB 2.5 MB/s eta 0:00:03
   -------------------- ------------------- 5.8/11.5 MB 2.5 MB/s eta 0:00:03
   --------------------- ------------------ 6.3/11.5 MB 2.5 MB/s eta 0:00:03
   ---

In [24]:
from CBFV import composition

c:\Users\胡扬\AppData\Local\Programs\Python\Python312\Lib\site-packages\CBFV\composition.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [25]:
x, y, formulae, skipped = composition.generate_features(cbfv_df)

Processing Input Data: 100%|██████████| 173/173 [00:00<00:00, 6864.72it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 173/173 [00:00<00:00, 4580.24it/s]


	Creating Pandas Objects...


In [26]:
x

,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,avg_families,avg_Metal,avg_Nonmetal,avg_Metalliod,avg_Mendeleev_Number,avg_l_quantum_number,...,mode_polarizability(A^3),mode_Melting_point_(K),mode_Boiling_Point_(K),mode_Density_(g/mL),mode_specific_heat_(J/g_K)_,mode_heat_of_fusion_(kJ/mol)_,mode_heat_of_vaporization_(kJ/mol)_,mode_thermal_conductivity_(W/(m_K))_,mode_heat_atomization(kJ/mol),mode_Cohesive_energy
0,10.650,22.693048,2.475,9.850,4.7125,0.5,0.5,0.0,55.8875,0.975,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
1,10.300,21.916634,2.450,9.700,4.6750,0.5,0.5,0.0,55.0250,0.950,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
2,9.950,21.140219,2.425,9.550,4.6375,0.5,0.5,0.0,54.1625,0.925,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
3,9.600,20.363805,2.400,9.400,4.6000,0.5,0.5,0.0,53.3000,0.900,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
4,9.575,20.290256,2.400,9.375,4.6000,0.5,0.5,0.0,53.2250,0.900,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,11.250,23.696700,2.500,10.250,4.7500,0.5,0.5,0.0,57.5000,1.000,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
169,10.875,23.086600,2.500,9.875,4.7500,0.5,0.5,0.0,56.3750,1.000,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
170,10.875,23.086600,2.500,9.875,4.7500,0.5,0.5,0.0,56.3750,1.000,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
171,10.875,23.086600,2.500,9.875,4.7500,0.5,0.5,0.0,56.3750,1.000,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62


In [27]:
num_rows = len(cbfv_df)
print(f'There are in total {num_rows} rows in the X_train DataFrame.')

num_unique_formulae = len(cbfv_df['formula'].unique())
print(f'But there are only {num_unique_formulae} unique formulae!\n')

There are in total 173 rows in the X_train DataFrame.
But there are only 115 unique formulae!



In [28]:
X = drx_df.drop(columns=["DRX label"]).copy()
y = drx_df["DRX label"].copy()

unique_formulae = X["Composition"].unique()

print(f"{len(unique_formulae)} unique formulae:\n{unique_formulae}")

115 unique formulae:
['Li1.05Mn0.85Ti0.1O2' 'Li1.1Mn0.7Ti0.2O2' 'Li1.15Mn0.55Ti0.3O2'
 'Li1.2Mn0.4Ti0.4O2' 'Li1.2Cr0.1Mn0.3Ti0.4O2' 'Li1.2Cr0.2Mn0.2Ti0.4O2'
 'Li1.2Ni0.2Ti0.6O2' 'Li1.2Cr0.1Ni0.15Ti0.55O2' 'Li1.2Cr0.2Ni0.1Ti0.5O2'
 'Li1.3Mn0.4Nb0.3O2' 'Li1.2Mn0.6Nb0.2O2' 'Li1.2Mo0.133Ni0.333Ti0.333O2'
 'Li1.14Mn0.57Ti0.29O2' 'Li1.3Fe0.4Nb0.3O2' 'Li1.3Nb0.3V0.4O2'
 'Li1.05Mo0.033Ni0.458Ti0.458O2' 'Li1.1Mo0.067Ni0.417Ti0.417O2'
 'Li1.15Mo0.1Ni0.375Ti0.375O2' 'Li1.333Mo0.333Ni0.333O2'
 'Li1.42Fe0.29Mo0.29O2' 'Li1.286Mo0.429Nb0.286O2' 'Li1.2Mo0.4Ti0.4O2'
 'Li1.333Mo0.667O2' 'Li1.333V0.667O2' 'LiCo0.5Ti0.5O2'
 'LiFe0.5Ti0.4V0.1O2' 'LiFe0.5Ti0.25V0.25O2' 'Li1.25Mn0.4Nb0.15Ti0.2O2'
 'LiNi0.5Ti0.5O2' 'Li1.05Ni0.458Ti0.458W0.033O2'
 'Li1.1Ni0.417Ti0.417W0.067O2' 'Li1.15Ni0.375Ti0.375W0.1O2'
 'Li1.2Ni0.333Ti0.333W0.133O2' 'Li1.143Mn0.286V0.572O2' 'Li1.2Mn0.2V0.6O2'
 'LiFeO2' 'Li1.04Fe0.88Ti0.08O2' 'Li1.09Fe0.73Ti0.18O2'
 'Li1.13Fe0.61Ti0.26O2' 'Li1.19Fe0.43Ti0.38O2' 'Li1.21Fe0.37Ti0.42O2'
 'Li1.2

In [29]:
import json

# Reuse the exact composition-level split used for the dissertation results.
# This frozen manifest preserves the original 80/23/12 formula split.
with open("data_splits/split_manifest.json", "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

train_formulae = split_manifest["train_formulae"]
val_formulae = split_manifest["validation_formulae"]
test_formulae = split_manifest["test_formulae"]

train_formula_set = set(train_formulae)
val_formula_set = set(val_formulae)
test_formula_set = set(test_formulae)

if train_formula_set & val_formula_set or train_formula_set & test_formula_set or val_formula_set & test_formula_set:
    raise ValueError("Formula overlap detected between data splits.")

all_split_formulae = train_formula_set | val_formula_set | test_formula_set
if all_split_formulae != set(unique_formulae):
    missing = set(unique_formulae) - all_split_formulae
    unexpected = all_split_formulae - set(unique_formulae)
    raise ValueError(f"Split manifest does not match the dataset. Missing: {missing}; unexpected: {unexpected}")

print("Number of training formulae:", len(train_formulae))
print("Number of validation formulae:", len(val_formulae))
print("Number of testing formulae:", len(test_formulae))

Number of training formulae: 80
Number of validation formulae: 23
Number of testing formulae: 12


In [31]:
# Split the original dataset into train / validation / test datasets
# using the Composition lists above

df_train = drx_df[drx_df["Composition"].isin(train_formulae)].copy()
df_val = drx_df[drx_df["Composition"].isin(val_formulae)].copy()
df_test = drx_df[drx_df["Composition"].isin(test_formulae)].copy()

print(f"train dataset shape: {df_train.shape}")
print(f"validation dataset shape: {df_val.shape}")
print(f"test dataset shape: {df_test.shape}\n")

print(df_train.head(), "\n")
print(df_val.head(), "\n")
print(df_test.head(), "\n")

train dataset shape: (128, 5)
validation dataset shape: (31, 5)
test dataset shape: (14, 5)

           Composition  DRX label  Method  t (h)  T (°C)
0  Li1.05Mn0.85Ti0.1O2          0       4   0.33  1100.0
1    Li1.1Mn0.7Ti0.2O2          0       4   0.33  1100.0
2  Li1.15Mn0.55Ti0.3O2          0       4   0.33  1100.0
3    Li1.2Mn0.4Ti0.4O2          1       4  10.00  1000.0
6    Li1.2Ni0.2Ti0.6O2          1       4   3.00  1000.0 

                 Composition  DRX label  Method  t (h)  T (°C)
4     Li1.2Cr0.1Mn0.3Ti0.4O2          1       4   10.0  1000.0
5     Li1.2Cr0.2Mn0.2Ti0.4O2          1       4   10.0  1050.0
23           Li1.333V0.667O2          0       1   14.0    25.0
24            LiCo0.5Ti0.5O2          0       3   10.0   660.0
27  Li1.25Mn0.4Nb0.15Ti0.2O2          1       2   12.0   950.0 

                     Composition  DRX label  Method  t (h)  T (°C)
21             Li1.2Mo0.4Ti0.4O2          1       1   36.0    25.0
26          LiFe0.5Ti0.25V0.25O2          1      

In [ ]:
train_formulae = set(df_train['Composition'].unique())
val_formulae = set(df_val['Composition'].unique())
test_formulae = set(df_test['Composition'].unique())

common_formulae1 = train_formulae.intersection(test_formulae)
common_formulae2 = train_formulae.intersection(val_formulae)
common_formulae3 = test_formulae.intersection(val_formulae)

print(f'# of common formulae in intersection 1: {len(common_formulae1)}; common formulae: {common_formulae1}')
print(f'# of common formulae in intersection 2: {len(common_formulae2)}; common formulae: {common_formulae2}')
print(f'# of common formulae in intersection 3: {len(common_formulae3)}; common formulae: {common_formulae3}')

In [32]:
import os

# Save these splits into csv files
PATH = os.getcwd()

# Create a folder to store the split datasets
split_dir = os.path.join(PATH, "data_splits")
os.makedirs(split_dir, exist_ok=True)

train_path = os.path.join(split_dir, "drx_train.csv")
val_path = os.path.join(split_dir, "drx_val.csv")
test_path = os.path.join(split_dir, "drx_test.csv")

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print("Saved train data to:", train_path)
print("Saved validation data to:", val_path)
print("Saved test data to:", test_path)

Saved train data to: c:\Users\胡扬\Downloads\NSCI0017-DRX-dissertation-main\Thesis_Supplementary_Files_Final\data_splits\drx_train.csv
Saved validation data to: c:\Users\胡扬\Downloads\NSCI0017-DRX-dissertation-main\Thesis_Supplementary_Files_Final\data_splits\drx_val.csv
Saved test data to: c:\Users\胡扬\Downloads\NSCI0017-DRX-dissertation-main\Thesis_Supplementary_Files_Final\data_splits\drx_test.csv


In [33]:
import numpy as np 


# =====================================================
# 0. Load the saved split files
# =====================================================

df_train = pd.read_csv("data_splits/drx_train.csv")
df_val = pd.read_csv("data_splits/drx_val.csv")
df_test = pd.read_csv("data_splits/drx_test.csv")

print("Loaded saved split files:")
print("df_train:", df_train.shape)
print("df_val:", df_val.shape)
print("df_test:", df_test.shape)


Loaded saved split files:
df_train: (128, 5)
df_val: (31, 5)
df_test: (14, 5)


In [34]:

# =====================================================
# 0.1 Featurise each split
# =====================================================

def featurise_split(df_split):
    cbfv_df = df_split[["Composition", "DRX label"]].copy()

    cbfv_df = cbfv_df.rename(
        columns={
            "Composition": "formula",
            "DRX label": "target"
        }
    )
# using oliynyk
    X_split, y_split, formulae_split, skipped_split = composition.generate_features(cbfv_df)

    X_split = X_split.reset_index(drop=True)
    y_split = pd.Series(y_split).reset_index(drop=True)

    return X_split, y_split, formulae_split, skipped_split


X_train, y_train, formulae_train, skipped_train = featurise_split(df_train)
X_val, y_val, formulae_val, skipped_val = featurise_split(df_val)
X_test, y_test, formulae_test, skipped_test = featurise_split(df_test)

print("Original X_train shape:", X_train.shape)
print("Original X_val shape:", X_val.shape)
print("Original X_test shape:", X_test.shape)


Processing Input Data: 100%|██████████| 128/128 [00:00<00:00, 11405.80it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 128/128 [00:00<00:00, 6001.91it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 31/31 [00:00<00:00, 10250.98it/s]


	Featurizing Compositions...


... (2 outputs left out)

Processing Input Data: 100%|██████████| 14/14 [00:00<00:00, 9149.31it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 14/14 [00:00<00:00, 5520.90it/s]


	Creating Pandas Objects...
Original X_train shape: (128, 264)
Original X_val shape: (31, 264)
Original X_test shape: (14, 264)


In [35]:
# =====================================================
# 0.2 Start from the newly featurised split data
# =====================================================

X_train_filtered = X_train.copy()
X_val_filtered = X_val.copy()
X_test_filtered = X_test.copy()


# Make sure y_train has the same index as X_train
y_train_for_filter = pd.Series(y_train).copy()

if len(y_train_for_filter) != len(X_train_filtered):
    raise ValueError("y_train length is not the same as X_train length.")

# If the index does not match, force it to match X_train
if not y_train_for_filter.index.equals(X_train_filtered.index):
    y_train_for_filter.index = X_train_filtered.index



In [36]:
# =====================================================
# 1. Remove low variance features
# =====================================================

variance_threshold = 0.0

numeric_cols = X_train_filtered.select_dtypes(include=["number"]).columns

x_num = X_train_filtered[numeric_cols].apply(pd.to_numeric, errors="coerce")

feature_variance = x_num.var(axis=0, skipna=True)

low_variance_features = []

for col in numeric_cols:
    v = float(feature_variance.get(col, np.nan))
    
    if (not np.isfinite(v)) or (v <= variance_threshold):
        low_variance_features.append(col)

# Drop the same low-variance features from train and test
X_train_filtered = X_train_filtered.drop(columns=low_variance_features, errors="ignore")
X_val_filtered = X_val_filtered.drop(columns=low_variance_features, errors="ignore")
X_test_filtered = X_test_filtered.drop(columns=low_variance_features, errors="ignore")

print("Low variance features removed:", len(low_variance_features))


Low variance features removed: 92


In [37]:
# =====================================================
# 2. Remove high correlation features
# =====================================================

correlation_threshold = 0.98

numeric_cols = X_train_filtered.select_dtypes(include=["number"]).columns

high_corr_records = []
high_correlation_features = []

if len(numeric_cols) >= 2:
    
    x_num = X_train_filtered[numeric_cols].apply(pd.to_numeric, errors="coerce")
    
    # Fill missing values using median from training data
    med = x_num.median(axis=0, skipna=True)
    x_num = x_num.fillna(med).fillna(0.0)
    
    # Calculate absolute correlation between features
    corr_abs = x_num.corr().abs().fillna(0.0)
    
    # Only use the upper triangle to avoid duplicate pairs
    upper = corr_abs.where(
        np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
    )
    
    # Calculate correlation between each feature and target
    y_num = pd.to_numeric(y_train_for_filter.reindex(x_num.index), errors="coerce")
    
    target_corr = {}
    
    for col in numeric_cols:
        xv = x_num[col].to_numpy(dtype=float)
        yv = y_num.to_numpy(dtype=float)
        
        mask = np.isfinite(xv) & np.isfinite(yv)
        
        if (
            int(np.sum(mask)) < 3
            or float(np.std(xv[mask])) < 1e-12
            or float(np.std(yv[mask])) < 1e-12
        ):
            target_corr[col] = float("nan")
        else:
            target_corr[col] = float(abs(np.corrcoef(xv[mask], yv[mask])[0, 1]))
    
    to_drop = set()
    sorted_pairs = upper.stack().sort_values(ascending=False)
    
    for (a, b), c in sorted_pairs.items():
        c_val = float(c)
        
        if c_val < correlation_threshold:
            break
        
        if a in to_drop or b in to_drop:
            continue
        
        ca = target_corr.get(str(a), float("nan"))
        cb = target_corr.get(str(b), float("nan"))
        
        # Keep the feature that has stronger correlation with the target
        if np.isnan(ca) and np.isnan(cb):
            drop_col = str(max(str(a), str(b)))
            keep_col = str(min(str(a), str(b)))
        elif np.isnan(ca):
            drop_col = str(a)
            keep_col = str(b)
        elif np.isnan(cb):
            drop_col = str(b)
            keep_col = str(a)
        elif ca < cb:
            drop_col = str(a)
            keep_col = str(b)
        elif cb < ca:
            drop_col = str(b)
            keep_col = str(a)
        else:
            drop_col = str(max(str(a), str(b)))
            keep_col = str(min(str(a), str(b)))
        
        to_drop.add(drop_col)
        
        high_corr_records.append({
            "feature_a": str(a),
            "feature_b": str(b),
            "abs_corr_train": c_val,
            "corr_with_target_a": ca,
            "corr_with_target_b": cb,
            "dropped_feature": drop_col,
            "kept_feature": keep_col,
            "threshold": correlation_threshold
        })
    
    high_correlation_features = sorted(to_drop)
    
    # Drop the same high-correlation features from train and test
    X_train_filtered = X_train_filtered.drop(columns=high_correlation_features, errors="ignore")
    X_val_filtered = X_val_filtered.drop(columns=high_correlation_features, errors="ignore")
    X_test_filtered = X_test_filtered.drop(columns=high_correlation_features, errors="ignore")

print("High correlation features removed:", len(high_correlation_features))


High correlation features removed: 66


In [38]:
# =====================================================
# 3. Make tables showing what was removed
# =====================================================

low_variance_removed = pd.DataFrame({
    "feature": low_variance_features,
    "train_variance": [feature_variance[f] for f in low_variance_features],
    "threshold": variance_threshold
})

high_correlation_removed = pd.DataFrame(high_corr_records)


# =====================================================
# 4. Save results
# =====================================================

low_variance_removed.to_csv("low_variance_removed.csv", index=False)
high_correlation_removed.to_csv("high_correlation_removed.csv", index=False)

X_train_filtered.to_csv("X_train_filtered.csv", index=False)
X_test_filtered.to_csv("X_test_filtered.csv", index=False)

print("Original X_train shape:", X_train.shape)
print("Filtered X_train shape:", X_train_filtered.shape)
print("Original X_test shape:", X_test.shape)
print("Filtered X_test shape:", X_test_filtered.shape)
print("Original X_val shape:", X_val.shape)
print("Filtered X_val shape:", X_val_filtered.shape)

Original X_train shape: (128, 264)
Filtered X_train shape: (128, 106)
Original X_test shape: (14, 264)
Filtered X_test shape: (14, 106)
Original X_val shape: (31, 264)
Filtered X_val shape: (31, 106)


In [39]:
high_correlation_removed

,feature_a,feature_b,abs_corr_train,corr_with_target_a,corr_with_target_b,dropped_feature,kept_feature,threshold
0,avg_Metal,avg_Number_of_unfilled_p_valence_electrons,1.000000,0.136392,0.136392,avg_Metal,avg_Number_of_unfilled_p_valence_electrons,0.98
1,avg_valence_p,avg_Number_of_unfilled_p_valence_electrons,1.000000,0.136392,0.136392,avg_valence_p,avg_Number_of_unfilled_p_valence_electrons,0.98
2,avg_Nonmetal,avg_Number_of_unfilled_p_valence_electrons,1.000000,0.136392,0.136392,avg_Nonmetal,avg_Number_of_unfilled_p_valence_electrons,0.98
3,min_heat_atomization(kJ/mol),min_Cohesive_energy,1.000000,0.200300,0.200300,min_heat_atomization(kJ/mol),min_Cohesive_energy,0.98
4,range_crystal_radius,min_crystal_radius,1.000000,0.084205,0.084205,range_crystal_radius,min_crystal_radius,0.98
...,...,...,...,...,...,...,...,...
61,range_heat_of_vaporization_(kJ/mol)_,max_Boiling_Point_(K),0.986736,0.162692,0.151020,max_Boiling_Point_(K),range_heat_of_vaporization_(kJ/mol)_,0.98
62,avg_families,dev_l_quantum_number,0.985388,0.007540,0.018899,avg_families,dev_l_quantum_number,0.98
63,avg_group,avg_number_of_valence_electrons,0.983621,0.047019,0.023114,avg_number_of_valence_electrons,avg_group,0.98
64,avg_crystal_radius,dev_crystal_radius,0.982110,0.014637,0.011544,dev_crystal_radius,avg_crystal_radius,0.98


In [40]:
print(X_train_filtered.shape)
print(X_val_filtered.shape)
print(X_test_filtered.shape)

(128, 106)
(31, 106)
(14, 106)


In [41]:
X_train_filtered.to_csv("X_train_filtered.csv", index=False)
X_val_filtered.to_csv("X_val_filtered.csv", index=False)
X_test_filtered.to_csv("X_test_filtered.csv", index=False)